https://github.com/FabricioArendTorres/FlowConductor/tree/main

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os
import numpy as np
import matplotlib.pyplot as plt

import torch
from msi.flow_conductor import architecture
from msi.flow_conductor.likelihood_flow import LikelihoodFlow

from msi.utils import preprocessing, plotting, diagnostics
from msfm.utils import prior, parameters, files, logger, observation

# load compressions

In [9]:
# use this cell as inspiration. You need arrays that resemble these.

base_dir = ""

# dataset
fidu_preds, grid_preds, grid_cosmos, file_dict = preprocessing.get_reshaped_network_preds(
    base_dir,
    model_dir,
    n_steps,
    with_fidu=False,
    # n_params=len(params), 
    # n_perms_per_cosmo=4
)

# you can use whatever paths you want

# output directory and file names
out_dir = os.path.join(base_dir, model_dir)

label = f"{n_steps}_steps_likelihood"
# label = f"{n_steps}_steps_posterior"

25-09-16 08:04:39 input_output INF   Loading predictions from /pscratch/sd/a/athomsen/run_files/v15/extended/combined/mutual_info/default/v1/preds_520000.h5 
25-09-16 08:04:39 input_output INF   Array shapes: 
25-09-16 08:04:39 input_output INF   fiducial/vali/pred = (40000, 20) 
25-09-16 08:04:39 input_output INF   fiducial/vali/i_example = (40000,) 
25-09-16 08:04:39 input_output INF   fiducial/vali/i_noise = (40000,) 
25-09-16 08:04:39 input_output INF   grid/pred          = (2500, 80, 20) 
25-09-16 08:04:39 input_output INF   grid/cosmo         = (2500, 80, 10) 
25-09-16 08:04:39 input_output INF   grid/i_example     = (2500, 80) 
25-09-16 08:04:39 input_output INF   grid/i_noise       = (2500, 80) 
25-09-16 08:04:39 input_output INF   grid/i_sobol       = (2500, 80) 


25-09-16 08:04:39 preprocessin INF   Shapes after concatenation and selection: 
25-09-16 08:04:39 preprocessin INF   grid_preds  = (200000, 20) 
25-09-16 08:04:39 preprocessin INF   grid_cosmos = (200000, 10) 


# likelihood Flow $p(x|\theta)$

### architecture

In [11]:
# input dimensions
x_dim = grid_preds.shape[-1]
theta_dim = grid_cosmos.shape[-1]

# shared hyperparameters
context_embedding_dim = 32

embedding_net = architecture.get_context_embedding_net(
    context_dim=theta_dim,
    context_embedding_dim=context_embedding_dim,
    hidden_dim=64,
    n_blocks=3,
    dropout_probability=0.0,
    use_batch_norm=False,
)    

base_dist = architecture.get_normal_dist(
    feature_dim=x_dim,
)

transform = architecture.get_sigmoids_transform(
    feature_dim=x_dim,
    context_embedding_dim=context_embedding_dim,
    n_layers=4,
    hidden_dim=256,
    svd_kwargs={},
    sigmoids_kwargs={
        "n_sigmoids": 16,
        "num_blocks": 3,
        "dropout_probability": 0.0,
    }
)

model = LikelihoodFlow(
    params, 
    conf, 
    embedding_net=embedding_net,
    base_dist=base_dist,
    transform=transform,
    out_dir=out_dir,
    # label=label,
    label=label + "_sigmoid",
    load_existing=True,
)

25-09-16 08:04:40 likelihood_b INF   Set up the model directory /pscratch/sd/a/athomsen/run_files/v15/extended/combined/mutual_info/default/v1/520000_steps_likelihood_sigmoid/likelihood_flow 
25-09-16 08:04:40 likelihood_f INF   Initialized the normalizing flow 
25-09-16 08:04:40 likelihood_f INF   Running on device cuda with default float torch.float32 
25-09-16 08:04:40 likelihood_f WAR   Could not load the model from /pscratch/sd/a/athomsen/run_files/v15/extended/combined/mutual_info/default/v1/520000_steps_likelihood_sigmoid/likelihood_flow/likelihood_flow.pt 


### training

In [1]:
n_cosmos = file_dict["grid/pred"].shape[0]
n_examples = grid_preds.shape[0]
# such that GPU utilization is maximized, but not larger
batch_size = 4 * n_cosmos
print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# default to train from scratch with 4 permutations per grid point
model.fit(
    x=grid_preds,
    theta=grid_cosmos,
    n_epochs=100,
    # dataset
    batch_size=batch_size,
    vali_split=0.1,
    # optimizer
    learning_rate=1e-3,
    weight_decay=0.0,
    clip_by_global_norm=1.0,
    # scheduler
    scheduler_type="cosine",
    scheduler_kwargs={"eta_min": 1e-6},
    # early stopping
    n_patience_epochs=None,
    min_delta=1e-5,
    save_model=True,
)

NameError: name 'file_dict' is not defined

# contours

## observations

In [ ]:
obs_dict = {}

### CosmoGrid internal

fiducial

In [ ]:
n_examples = 15

for i_fidu in range(n_examples):
    obs_dict[f"fiducial_{i_fidu}"] = {
        "pred": fidu_preds[i_fidu], 
        "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
    }


grid 

In [ ]:
n_examples = 4

for i_grid in range(n_examples):
    i_grid *= 80
    obs_dict[f"grid_{i_grid}"] = {
        "pred": grid_preds[i_grid],
        "point": {str(param): value for param, value in zip(params, grid_cosmos[i_grid])},
    }


### MCMC and plotting

In [ ]:
extra_label = ""

for key in obs_dict.keys():
    print(f"\nStarting with mock observation {key}")
    
    posterior_samples = model.sample_posterior(
        obs_dict[key]["pred"],
        label=key+extra_label,
        n_walkers=1024,
        n_burnin_steps=1000,
        n_samples=1024*1000,
    )

    model.plot_contours(
        posterior_samples,
        obs_point=obs_dict[key]["point"],
        obs_label=key,
        label=key+extra_label,
        with_des_chain=False,
        density=True,
    )